In [ ]:
# 日本語の合成音声を生成し、Botの /wav と同じWAV形式を確認します。
# 人物の録音、Discordトークン、外部の音声APIは使用しません。
import shutil, subprocess, tempfile, wave
from pathlib import Path
from IPython.display import Audio, display

def run(*args, **kwargs):
    return subprocess.run(args, check=True, **kwargs)

if not shutil.which('open_jtalk') or not list(Path('/usr/share').glob('**/nitech_jp_atr503_m001.htsvoice')):
    run('apt-get', '-qq', 'update')
    run('apt-get', '-qq', 'install', '-y', 'open-jtalk', 'hts-voice-nitech-jp-atr503-m001')
voice = next(Path('/usr/share').glob('**/nitech_jp_atr503_m001.htsvoice'))
run('apt-get', '-qq', 'install', '-y', 'open-jtalk-mecab-naist-jdic')
files = subprocess.check_output(['dpkg', '-L', 'open-jtalk-mecab-naist-jdic'], text=True).splitlines()
dictionary = next(Path(p).parent for p in files if p.endswith('/sys.dic'))
out = Path(tempfile.mkdtemp()) / 'speech.wav'
run('open_jtalk', '-x', str(dictionary), '-m', str(voice), '-ow', str(out), input='こんにちは。音声のテストです。', text=True)
b = out.read_bytes()
assert b[:4] == b'RIFF' and b[8:12] == b'WAVE'
with wave.open(str(out)) as w:
    assert w.getnframes() > 0
    print('WAV OK:', w.getframerate(), 'Hz /', len(b), 'bytes')
print('音声: Open JTalk / NIT ATR503 M001 (HTS Working Group, CC BY 3.0)')
display(Audio(filename=str(out)))